# 3.8 — Ridge Regression

Ridge regression is ordinary least squares with one extra promise: fit the data, but pay a squared-length charge for large coefficients. That L2 charge makes unstable directions expensive, so the model usually gives up a little training fit to gain a smoother, more reusable rule.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build ridge regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is kept small enough to inspect. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, linear algebra, and reproducible toy data.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # make random examples repeatable.

### 1. Empirical risk is the average squared miss

Ridge still begins with the empirical-risk idea: compare predictions with observed targets and average the losses. The loss we use here is squared error, so a miss of 2 costs four times a miss of 1. The lesson's verified toy losses are 0.268, 0.122, and 0.505; their average is the raw training score before any stability cost is added.

In [ ]:
losses_w = np.array([0.268, 0.122, 0.505])  # per-example squared losses from the lesson text.
raw_risk_w = float(losses_w.mean())  # empirical risk = average loss over the sample.
print("losses:", losses_w)
print("raw empirical risk:", round(raw_risk_w, 3))
assert round(raw_risk_w, 3) == 0.298

▶ What you'll see: the three losses average to 0.298, the raw fit number ridge starts from.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["ex1", "ex2", "ex3"], losses_w, color="steelblue")
plt.axhline(raw_risk_w, color="crimson", linestyle="--", label="mean risk")
plt.ylabel("squared loss")
plt.title("1: empirical risk averages example losses")
plt.legend()
plt.show()

▶ What you'll see: one dashed line summarizing the three individual squared errors.

*Why it's done this way:* empirical risk converts many prediction mistakes into one objective, and squaring makes big mistakes disproportionately visible. Ridge does not replace this fit term; it adds a stability term to it so model selection is based on the full cost, not the prettiest training fragment.

### 2. Ordinary least squares can become unstable

Least squares chooses coefficients that minimize squared residuals. When two columns of `X` are nearly the same, many coefficient pairs make almost the same predictions, so the fitted coefficients can become large and cancel each other. That is variance: the prediction rule is sensitive to tiny data changes even when the training error looks fine.

In [ ]:
n_w = 40
x1_w = np.linspace(-2, 2, n_w)
x2_w = x1_w + 0.03 * np.random.randn(n_w)  # almost a duplicate of x1.
X_w = np.column_stack([x1_w, x2_w])
y_w = 3 * x1_w + 0.2 * np.random.randn(n_w)
print("correlation between columns:", round(float(np.corrcoef(X_w.T)[0, 1]), 4))

▶ What you'll see: the two features are almost perfectly correlated, which makes coefficient attribution fragile.

In [ ]:
beta_ols_w = np.linalg.solve(X_w.T @ X_w, X_w.T @ y_w)  # normal equation for OLS.
pred_ols_w = X_w @ beta_ols_w
mse_ols_w = float(np.mean((y_w - pred_ols_w) ** 2))
print("OLS coefficients:", np.round(beta_ols_w, 3))
print("OLS MSE:", round(mse_ols_w, 4))

▶ What you'll see: the two coefficients can be larger and more oppositely allocated than the simple story `y≈3*x1` suggests.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.scatter(x1_w, y_w, s=22, label="data")
plt.plot(x1_w, pred_ols_w, color="crimson", label="OLS fit")
plt.xlabel("x1")
plt.ylabel("y")
plt.title("2: good fit can hide unstable coefficients")
plt.legend()
plt.show()

▶ What you'll see: the fit line looks reasonable even though the two-feature coefficient vector is hard to trust.

*Why it's done this way:* least squares only asks for small residuals, so it has no reason to prefer a small, stable coefficient vector over a large, cancelling one when both predict similarly. Ridge adds exactly that missing preference.

### 3. The L2 penalty charges squared coefficient length

Ridge minimizes `||y - Xβ||² + λ||β||²`. The first term rewards fit; the second charges coefficient length. Because the penalty squares coefficients, doubling a coefficient quadruples its cost. The lesson's toy arithmetic adds a cost of 0.070 to the raw risk 0.298, producing the score 0.368.

In [ ]:
cost_w = 0.070
score_w = raw_risk_w + cost_w
print("raw risk:", round(raw_risk_w, 3))
print("ridge cost:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(score_w, 3) == 0.368

▶ What you'll see: the selection score is 0.368, not the raw 0.298.

In [ ]:
beta_grid_w = np.linspace(-3, 3, 121)
penalty_curve_w = beta_grid_w ** 2
plt.figure(figsize=(4.4, 3))
plt.plot(beta_grid_w, penalty_curve_w, color="purple")
plt.xlabel("one coefficient β")
plt.ylabel("β² penalty")
plt.title("3: L2 cost grows quadratically")
plt.show()

▶ What you'll see: the penalty is flat near zero and rises fast for large positive or negative coefficients.

*Why it's done this way:* the L2 term is a mathematical statement of distrust in unnecessarily large coefficients. It is smooth, symmetric, and differentiable, so it discourages size without creating the sharp zero-setting behavior of L1 lasso.

### 4. Ridge has a closed-form shrinkage solution

For a fixed `λ`, ridge has a normal-equation solution: `(XᵀX + λI)β = Xᵀy`. The added `λI` lifts the diagonal of `XᵀX`, making the linear system easier to solve and shrinking coefficients toward zero. We usually do not penalize the intercept, so this toy centers `X` and `y` first.

In [ ]:
Xc_w = X_w - X_w.mean(axis=0)
yc_w = y_w - y_w.mean()
lam_w = 1.0
beta_ridge_w = np.linalg.solve(Xc_w.T @ Xc_w + lam_w * np.eye(Xc_w.shape[1]), Xc_w.T @ yc_w)
print("ridge coefficients λ=1:", np.round(beta_ridge_w, 3))
print("OLS length:", round(float(np.linalg.norm(beta_ols_w)), 3), "ridge length:", round(float(np.linalg.norm(beta_ridge_w)), 3))
assert np.linalg.norm(beta_ridge_w) < np.linalg.norm(beta_ols_w)

▶ What you'll see: ridge uses a shorter coefficient vector than OLS on the same data.

In [ ]:
pred_ridge_w = (X_w - X_w.mean(axis=0)) @ beta_ridge_w + y_w.mean()
plt.figure(figsize=(4.6, 3))
plt.scatter(x1_w, y_w, s=22, color="gray", label="data")
plt.plot(x1_w, pred_ols_w, color="crimson", label="OLS")
plt.plot(x1_w, pred_ridge_w, color="seagreen", label="ridge λ=1")
plt.xlabel("x1")
plt.ylabel("y")
plt.title("4: similar predictions, smaller coefficients")
plt.legend()
plt.show()

▶ What you'll see: ridge and OLS predictions are close, but ridge bought that fit with less coefficient length.

*Why it's done this way:* adding `λI` changes the geometry of the least-squares bowl: flat, unstable directions become more curved, so the optimum cannot wander far along them without paying penalty. Centering keeps the intercept from being shrunk merely because the target has a nonzero mean.

### 5. The λ knob trades fit for stability

When `λ=0`, ridge is OLS. As `λ` grows, the model pays more attention to coefficient length and less to squeezing out every last bit of training fit. The correct question is not whether training error always improves — it usually does not — but whether validation or future performance becomes more stable.

In [ ]:
lams_w = np.array([0.0, 0.01, 0.1, 1.0, 10.0, 100.0])
coef_path_w = []
train_mse_w = []
for lam_i_w in lams_w:
    beta_i_w = np.linalg.solve(Xc_w.T @ Xc_w + lam_i_w * np.eye(2), Xc_w.T @ yc_w)
    coef_path_w.append(beta_i_w)
    pred_i_w = Xc_w @ beta_i_w + y_w.mean()
    train_mse_w.append(float(np.mean((y_w - pred_i_w) ** 2)))
coef_path_w = np.array(coef_path_w)
print("λ grid:", lams_w)
print("coefficient lengths:", np.round(np.linalg.norm(coef_path_w, axis=1), 3))

▶ What you'll see: coefficient length steadily shrinks as λ increases.

In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogx(lams_w + 1e-6, coef_path_w[:, 0], marker="o", label="β1")
plt.semilogx(lams_w + 1e-6, coef_path_w[:, 1], marker="o", label="β2")
plt.xlabel("λ")
plt.ylabel("coefficient value")
plt.title("5: ridge coefficient path")
plt.legend()
plt.show()

▶ What you'll see: both coefficients are pulled toward zero as the penalty gets stronger.

In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogx(lams_w + 1e-6, train_mse_w, marker="o", color="darkorange")
plt.xlabel("λ")
plt.ylabel("training MSE")
plt.title("5: training fit usually worsens as λ grows")
plt.show()

▶ What you'll see: training error is lowest near λ=0, reminding us that ridge is chosen for generalization, not raw training vanity.

*Why it's done this way:* λ is a contract about how much coefficient length is worth. Small λ trusts the data enough to fit flexibly; large λ says the sample is too fragile to justify large weights. Validation decides which contract survives on unseen examples.

### 6. Feature scale controls how the penalty feels

The penalty is applied to coefficients, not directly to raw features. If one feature is measured in tiny units and another in huge units, the same predictive effect can require very different coefficient sizes. Ridge should therefore standardize non-intercept features before penalizing them.

In [ ]:
x_small_w = np.linspace(0, 1, 30)
x_large_w = 1000 * x_small_w
X_scale_w = np.column_stack([x_small_w, x_large_w])
y_scale_w = 2 * x_small_w + 0.05 * np.random.randn(30)
print("feature stds:", np.round(X_scale_w.std(axis=0), 3))

▶ What you'll see: the second feature's scale is 1000 times larger, so raw coefficients are not comparable.

In [ ]:
X_bad_w = X_scale_w - X_scale_w.mean(axis=0)
y_bad_w = y_scale_w - y_scale_w.mean()
beta_bad_w = np.linalg.solve(X_bad_w.T @ X_bad_w + 1.0 * np.eye(2), X_bad_w.T @ y_bad_w)
X_std_w = (X_scale_w - X_scale_w.mean(axis=0)) / X_scale_w.std(axis=0)
beta_good_w = np.linalg.solve(X_std_w.T @ X_std_w + 1.0 * np.eye(2), X_std_w.T @ y_bad_w)
print("ridge on raw scales:", np.round(beta_bad_w, 6))
print("ridge on standardized features:", np.round(beta_good_w, 3))

▶ What you'll see: raw-scale coefficients are dominated by units; standardized coefficients live on a fair scale.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["raw β1", "raw β2"], np.abs(beta_bad_w), color="crimson")
plt.title("6: raw coefficient sizes reflect units")
plt.ylabel("absolute coefficient")
plt.show()

▶ What you'll see: coefficient magnitude alone is misleading when feature units differ.

*Why it's done this way:* ridge's penalty assumes a one-unit coefficient means the same kind of model complexity across features. Standardization makes that assumption approximately true by putting predictors on comparable scales.

### 7. Validation chooses the full decision score

The lesson's arithmetic compares a baseline score 0.368, a more flexible alternative 0.416, and a stabilized score 0.294. The lower full score wins, and the relative gap tells us whether the win is meaningfully large rather than just numerically lower.

In [ ]:
baseline_w = 0.368
flexible_w = 0.416
stable_w = 0.80 * baseline_w
gap_w = flexible_w - baseline_w
relative_gap_w = gap_w / flexible_w
print("baseline:", baseline_w, "flexible:", flexible_w, "stable:", round(stable_w, 3))
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
assert round(stable_w, 3) == 0.294
assert round(gap_w, 3) == 0.048

▶ What you'll see: the stabilized score 0.294 is lower than both alternatives.

In [ ]:
scores_w = np.array([baseline_w, flexible_w, stable_w])
labels_w = ["baseline", "flexible", "stabilized"]
plt.figure(figsize=(4.8, 3))
plt.bar(labels_w, scores_w, color=["gray", "crimson", "seagreen"])
plt.ylabel("full decision score")
plt.title("7: choose by full score, not raw fit")
plt.show()

▶ What you'll see: the green stabilized bar is the smallest, so it is the toy winner.

*Why it's done this way:* model selection must compare quantities on the same scale: raw fit plus the relevant complexity or validation cost. Ridge is valuable only when the stability it buys is worth the fit it gives up.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, linear algebra, random numbers, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for plots that make coefficients, errors, and validation curves inspectable.
np.random.seed(0) # make all randomized examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Average per-example losses

**Goal.** Compute empirical risk from individual losses, because ridge starts with the same averaged training-loss frame as ordinary least squares.

In [ ]:
losses_b1 = np.array([0.268, 0.122, 0.505]) # store the verified toy per-example losses.
risk_b1 = float(np.mean(losses_b1)) # average the losses into empirical risk.
print("empirical risk:", round(risk_b1, 3)) # inspect the raw training score.
assert round(risk_b1, 3) == 0.298 # verify the lesson arithmetic.

▶ What you'll see: the raw empirical risk is 0.298.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact bar chart.
plt.bar(["1", "2", "3"], losses_b1, color="steelblue") # draw one bar per example loss.
plt.axhline(risk_b1, color="red", linestyle="--") # mark the average risk.
plt.title("Basic 1: individual losses and their mean") # title the plot.
plt.ylabel("loss") # label the loss axis.
plt.show() # display the chart.

▶ What you'll see: the mean line sits between the three losses.

👀 Takeaway: ridge's fit term is still an average of prediction mistakes.

### Basic 2 — Add the ridge cost

**Goal.** Add a regularization cost to raw risk, because model selection should use the full ridge objective scale.

In [ ]:
risk_b2 = 0.298 # use the verified raw empirical risk.
cost_b2 = 0.070 # use the verified stability or complexity cost.
score_b2 = risk_b2 + cost_b2 # compute the full selection score.
print("score:", round(score_b2, 3)) # inspect the full score.
assert round(score_b2, 3) == 0.368 # verify the lesson number.

▶ What you'll see: the score becomes 0.368 after adding the penalty cost.

In [ ]:
plt.figure(figsize=(4, 3)) # create a small stacked-cost figure.
plt.bar(["risk", "cost", "total"], [risk_b2, cost_b2, score_b2], color=["gray", "orange", "seagreen"]) # show the pieces and total.
plt.title("Basic 2: full ridge score") # title the plot.
plt.ylabel("score component") # label the score scale.
plt.show() # display the plot.

▶ What you'll see: the total bar is risk plus cost.

👀 Takeaway: the raw training number is incomplete without the regularization term.

### Basic 3 — Compute an L2 coefficient length

**Goal.** Measure coefficient length, because ridge charges squared L2 norm `||β||²`.

In [ ]:
beta_b3 = np.array([2.0, -1.0, 0.5]) # define a small coefficient vector.
squares_b3 = beta_b3 ** 2 # square each coefficient for the L2 penalty.
l2_sq_b3 = float(np.sum(squares_b3)) # sum squared coefficients.
print("squared pieces:", squares_b3) # inspect each contribution.
print("||beta||^2:", l2_sq_b3) # inspect the total penalty base.
assert l2_sq_b3 == 5.25 # verify 4 + 1 + 0.25.

▶ What you'll see: larger coefficients dominate the squared-length charge.

In [ ]:
plt.figure(figsize=(4, 3)) # create a coefficient-penalty chart.
plt.bar(["β1²", "β2²", "β3²"], squares_b3, color="purple") # show squared contributions.
plt.title("Basic 3: L2 penalty pieces") # title the plot.
plt.ylabel("squared coefficient") # label the squared scale.
plt.show() # display the chart.

▶ What you'll see: β1 contributes most because squaring magnifies large magnitude.

👀 Takeaway: ridge discourages large coefficient length smoothly and symmetrically.

### Basic 4 — Build a simple linear design matrix

**Goal.** Create a small `X` and `y`, because ridge fits coefficients in a linear prediction rule `Xβ`.

In [ ]:
x_b4 = np.array([-2., -1., 0., 1., 2.]) # define one input feature.
X_b4 = np.column_stack([x_b4]) # make it a 2-D design matrix with one column.
y_b4 = 1.0 + 2.0 * x_b4 # create exact targets from an intercept plus slope.
print("X shape:", X_b4.shape) # inspect design-matrix dimensions.
print("y:", y_b4) # inspect targets.

▶ What you'll see: five examples with one predictor column.

In [ ]:
plt.figure(figsize=(4, 3)) # create a tiny scatter plot.
plt.scatter(x_b4, y_b4, color="teal") # show the linear data.
plt.title("Basic 4: one-feature linear data") # title the plot.
plt.xlabel("x") # label the predictor axis.
plt.ylabel("y") # label the target axis.
plt.show() # display the scatter.

▶ What you'll see: the points lie exactly on a line.

👀 Takeaway: ridge changes how coefficients are chosen, not the basic linear prediction form.

### Basic 5 — Center before penalizing

**Goal.** Center `X` and `y`, because ridge usually leaves the intercept unpenalized and shrinks only slopes.

In [ ]:
def center_xy(X, y): # center predictors and target so ridge can leave the intercept unpenalized.
    X = np.asarray(X, dtype=float) # convert predictors to a floating-point array.
    y = np.asarray(y, dtype=float) # convert targets to a floating-point array.
    return X - X.mean(axis=0), y - y.mean(), X.mean(axis=0), float(y.mean()) # return centered data and the means.

def ridge_beta(X, y, lam): # solve the centered ridge normal equation.
    Xc, yc, _, _ = center_xy(X, y) # center before penalizing slopes.
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc) # solve (XᵀX+λI)β=Xᵀy.

def ridge_predict(X, beta, X_mean, y_mean): # rebuild predictions with the unpenalized intercept.
    return (np.asarray(X, dtype=float) - X_mean) @ beta + y_mean # apply centered slopes and add y mean.

def mse(y, pred): # compute mean squared error for compact checks.
    return float(np.mean((np.asarray(y, dtype=float) - np.asarray(pred, dtype=float)) ** 2)) # average squared residuals.

Xc_b5, yc_b5, Xm_b5, ym_b5 = center_xy(X_b4, y_b4) # center the one-feature data from Basic 4.
print("X mean:", Xm_b5, "y mean:", ym_b5) # inspect the intercept information.
print("centered y:", yc_b5) # inspect target values after removing the mean.

▶ What you'll see: centered values have mean zero.

In [ ]:
print("centered X mean:", np.round(Xc_b5.mean(axis=0), 6)) # verify predictor centering.
print("centered y mean:", round(float(yc_b5.mean()), 6)) # verify target centering.
plt.figure(figsize=(4, 3)) # create a centered-data plot.
plt.scatter(Xc_b5[:, 0], yc_b5, color="darkorange") # show centered coordinates.
plt.axhline(0, color="gray", linewidth=0.8) # mark zero target.
plt.axvline(0, color="gray", linewidth=0.8) # mark zero predictor.
plt.title("Basic 5: centered training data") # title the plot.
plt.show() # display the centered scatter.

▶ What you'll see: the same relationship now passes through the origin.

👀 Takeaway: centering separates the intercept from the coefficients ridge penalizes.

### Basic 6 — Solve ordinary least squares

**Goal.** Fit the unregularized slope, because ridge with λ=0 reduces to OLS.

In [ ]:
beta_ols_b6 = ridge_beta(X_b4, y_b4, 0.0) # solve centered least squares through the ridge helper with lambda zero.
print("OLS slope:", beta_ols_b6) # inspect the recovered slope.
assert round(float(beta_ols_b6[0]), 3) == 2.0 # verify the exact slope.

▶ What you'll see: the fitted slope is 2.0.

In [ ]:
pred_b6 = ridge_predict(X_b4, beta_ols_b6, Xm_b5, ym_b5) # rebuild predictions with the intercept mean.
print("predictions:", pred_b6) # inspect predictions.
assert mse(y_b4, pred_b6) == 0.0 # verify exact fit.
plt.figure(figsize=(4, 3)) # create a fit plot.
plt.scatter(x_b4, y_b4, color="gray") # show data.
plt.plot(x_b4, pred_b6, color="crimson") # show OLS predictions.
plt.title("Basic 6: OLS exact fit") # title the plot.
plt.show() # display the fit.

▶ What you'll see: the prediction line passes through every point.

👀 Takeaway: λ=0 is the unregularized least-squares endpoint.

### Basic 7 — Solve ridge with one feature

**Goal.** Fit a ridge slope with λ>0, because the penalty shrinks the OLS coefficient toward zero.

In [ ]:
lam_b7 = 5.0 # choose a visible regularization strength.
beta_ridge_b7 = ridge_beta(X_b4, y_b4, lam_b7) # solve ridge on centered one-feature data.
print("ridge slope:", round(float(beta_ridge_b7[0]), 3)) # inspect the shrunk slope.
assert round(float(beta_ridge_b7[0]), 3) == 1.333 # verify 20 / (10 + 5).

▶ What you'll see: the ridge slope is 1.333, smaller than the OLS slope 2.0.

In [ ]:
pred_b7 = ridge_predict(X_b4, beta_ridge_b7, Xm_b5, ym_b5) # make ridge predictions with the unpenalized intercept.
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.scatter(x_b4, y_b4, color="gray", label="data") # show data.
plt.plot(x_b4, pred_b6, color="crimson", label="OLS") # show OLS line.
plt.plot(x_b4, pred_b7, color="seagreen", label="ridge") # show ridge line.
plt.title("Basic 7: shrinkage flattens the line") # title the plot.
plt.legend() # show labels.
plt.show() # display the comparison.

▶ What you'll see: ridge keeps the same center but flattens the slope.

👀 Takeaway: ridge trades exact training fit for a smaller coefficient.

### Basic 8 — Inspect residuals after shrinkage

**Goal.** Compare OLS and ridge residuals, because shrinkage deliberately accepts some training error.

In [ ]:
resid_ols_b8 = y_b4 - pred_b6 # compute OLS residuals.
resid_ridge_b8 = y_b4 - pred_b7 # compute ridge residuals.
print("OLS MSE:", mse(y_b4, pred_b6)) # inspect unregularized training error.
print("ridge MSE:", round(mse(y_b4, pred_b7), 3)) # inspect ridge training error.
assert round(mse(y_b4, pred_b7), 3) == 0.889 # verify the shrinkage error.

▶ What you'll see: OLS has zero training MSE while ridge has nonzero MSE.

In [ ]:
plt.figure(figsize=(4, 3)) # create a residual comparison.
plt.bar(["OLS", "ridge"], [np.linalg.norm(resid_ols_b8), np.linalg.norm(resid_ridge_b8)], color=["gray", "seagreen"]) # compare residual lengths.
plt.title("Basic 8: shrinkage costs training fit") # title the plot.
plt.ylabel("residual L2 length") # label residual size.
plt.show() # display the bar chart.

▶ What you'll see: ridge has a larger residual norm on the training set.

👀 Takeaway: ridge is not chosen to minimize training error alone; it is chosen to stabilize future predictions.

### Basic 9 — Compute the full ridge objective

**Goal.** Add squared-error and penalty terms explicitly, because ridge minimizes their sum.

In [ ]:
sse_b9 = float(np.sum(resid_ridge_b8 ** 2)) # compute total squared error for the ridge fit.
penalty_b9 = lam_b7 * float(np.sum(beta_ridge_b7 ** 2)) # compute lambda times coefficient squared length.
objective_b9 = sse_b9 + penalty_b9 # add fit and penalty terms.
print("SSE:", round(sse_b9, 3), "penalty:", round(penalty_b9, 3), "objective:", round(objective_b9, 3)) # inspect pieces.
assert round(objective_b9, 3) == 13.333 # verify the one-feature ridge objective.

▶ What you'll see: the objective is fit error plus coefficient-size cost.

In [ ]:
plt.figure(figsize=(4, 3)) # create an objective breakdown chart.
plt.bar(["SSE", "λ||β||²", "total"], [sse_b9, penalty_b9, objective_b9], color=["orange", "purple", "seagreen"]) # compare terms.
plt.title("Basic 9: ridge objective pieces") # title the plot.
plt.ylabel("objective units") # label objective scale.
plt.show() # display the plot.

▶ What you'll see: the penalty is a real part of the minimized quantity.

👀 Takeaway: ridge's optimum balances two forces, not one.

### Basic 10 — Compare full scores for model selection

**Goal.** Reproduce the lesson's baseline, flexible, and stabilized comparison, because the smallest full score is the chosen model.

In [ ]:
scores_b10 = np.array([0.368, 0.416, 0.294]) # baseline, flexible alternative, stabilized ridge score.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # label the candidates.
best_b10 = labels_b10[int(np.argmin(scores_b10))] # choose the lowest score.
print("best candidate:", best_b10) # inspect the winner.
assert best_b10 == "stabilized" # verify the lesson decision.

▶ What you'll see: the stabilized model wins the toy comparison.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a model-selection chart.
plt.bar(labels_b10, scores_b10, color=["gray", "crimson", "seagreen"]) # plot candidate scores.
plt.title("Basic 10: lower full score wins") # title the plot.
plt.ylabel("decision score") # label score scale.
plt.show() # display the chart.

▶ What you'll see: the stabilized bar is shortest.

👀 Takeaway: compare ridge models on the same decision scale: fit plus stability cost or validation evidence.

## 🟡 Easy

### Easy 1 — Ridge handles correlated predictors

**Goal.** Fit OLS and ridge on nearly duplicated columns, because ridge is especially useful when least-squares coefficients are unstable.

In [ ]:
def center_xy(X, y): # center predictors and target inside the Easy section.
    X = np.asarray(X, dtype=float) # convert predictors to floats.
    y = np.asarray(y, dtype=float) # convert targets to floats.
    return X - X.mean(axis=0), y - y.mean(), X.mean(axis=0), float(y.mean()) # return centered data and means.

def ridge_beta(X, y, lam): # solve centered ridge for the Easy examples.
    Xc, yc, _, _ = center_xy(X, y) # center slopes and target.
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc) # closed-form coefficients.

def ridge_predict(X, beta, X_mean, y_mean): # predict with centered ridge coefficients.
    return (np.asarray(X, dtype=float) - X_mean) @ beta + y_mean # add the intercept back.

def mse(y, pred): # compute mean squared error for validation.
    return float(np.mean((np.asarray(y, dtype=float) - np.asarray(pred, dtype=float)) ** 2)) # average squared error.

x_e1 = np.linspace(-2, 2, 50) # create a smooth base feature.
X_e1 = np.column_stack([x_e1, x_e1 + 0.02 * np.random.randn(50)]) # add a nearly duplicate feature.
y_e1 = 3 * x_e1 + 0.2 * np.random.randn(50) # create noisy targets from the shared signal.
print("feature correlation:", round(float(np.corrcoef(X_e1.T)[0, 1]), 4)) # inspect collinearity.

▶ What you'll see: the predictors are almost perfectly correlated.

In [ ]:
beta_ols_e1 = ridge_beta(X_e1, y_e1, 0.0) # fit OLS.
beta_ridge_e1 = ridge_beta(X_e1, y_e1, 1.0) # fit ridge with lambda one.
print("OLS beta:", np.round(beta_ols_e1, 3)) # inspect unstable allocation.
print("ridge beta:", np.round(beta_ridge_e1, 3)) # inspect shrunken allocation.
assert np.linalg.norm(beta_ridge_e1) < np.linalg.norm(beta_ols_e1) # verify shrinkage.

In [ ]:
plt.figure(figsize=(4, 3)) # create a coefficient comparison.
plt.bar(["OLS", "ridge"], [np.linalg.norm(beta_ols_e1), np.linalg.norm(beta_ridge_e1)], color=["crimson", "seagreen"]) # compare coefficient lengths.
plt.title("Easy 1: ridge shortens correlated coefficients") # title the plot.
plt.ylabel("||β||") # label coefficient length.
plt.show() # display the bar chart.

▶ What you'll see: ridge keeps a much shorter coefficient vector.

👀 Takeaway: ridge stabilizes coefficient estimates when features carry redundant information.

### Easy 2 — Sweep λ and plot coefficient paths

**Goal.** Trace coefficients across λ values, because ridge shrinkage is continuous rather than all-or-nothing.

In [ ]:
lams_e2 = np.array([0.0, 0.01, 0.1, 1.0, 10.0, 100.0]) # define a regularization grid.
coefs_e2 = np.array([ridge_beta(X_e1, y_e1, lam_e2) for lam_e2 in lams_e2]) # fit one ridge model per lambda.
lengths_e2 = np.linalg.norm(coefs_e2, axis=1) # compute coefficient length for each lambda.
print("lengths:", np.round(lengths_e2, 3)) # inspect shrinkage.
assert lengths_e2[-1] < lengths_e2[0] # verify high lambda shrinks more than OLS.

▶ What you'll see: coefficient length decreases as λ grows.

In [ ]:
plt.figure(figsize=(5, 3)) # create a coefficient-path plot.
plt.semilogx(lams_e2 + 1e-6, coefs_e2[:, 0], marker="o", label="β1") # plot first coefficient path.
plt.semilogx(lams_e2 + 1e-6, coefs_e2[:, 1], marker="o", label="β2") # plot second coefficient path.
plt.title("Easy 2: ridge path over λ") # title the plot.
plt.xlabel("λ") # label regularization axis.
plt.ylabel("coefficient") # label coefficient axis.
plt.legend() # show labels.
plt.show() # display the path.

▶ What you'll see: both coefficients move toward zero smoothly.

👀 Takeaway: λ is a continuous capacity knob for ridge.

### Easy 3 — Standardize before ridge

**Goal.** Compare raw-scale and standardized ridge coefficients, because the L2 penalty only makes fair comparisons when features share scale.

In [ ]:
x_e3 = np.linspace(0, 1, 40) # create a base feature.
X_e3 = np.column_stack([x_e3, 1000 * x_e3]) # create the same signal in very different units.
y_e3 = 2 * x_e3 + 0.05 * np.random.randn(40) # create targets from the base feature.
print("feature stds:", np.round(X_e3.std(axis=0), 3)) # inspect scale mismatch.

▶ What you'll see: the second feature is 1000 times larger in scale.

In [ ]:
beta_raw_e3 = ridge_beta(X_e3, y_e3, 1.0) # fit ridge on raw units.
Xstd_e3 = (X_e3 - X_e3.mean(axis=0)) / X_e3.std(axis=0) # standardize features.
beta_std_e3 = ridge_beta(Xstd_e3, y_e3, 1.0) # fit ridge after standardization.
print("raw beta:", np.round(beta_raw_e3, 6)) # inspect unit-dependent coefficients.
print("standardized beta:", np.round(beta_std_e3, 3)) # inspect comparable coefficients.
assert abs(beta_raw_e3[1]) < abs(beta_std_e3[1]) # verify unit scale changes coefficient magnitude.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a scale comparison plot.
plt.bar(["raw β1", "raw β2", "std β1", "std β2"], np.r_[np.abs(beta_raw_e3), np.abs(beta_std_e3)], color=["crimson", "crimson", "seagreen", "seagreen"]) # compare magnitudes.
plt.title("Easy 3: scale changes coefficient interpretation") # title the plot.
plt.ylabel("absolute coefficient") # label magnitude axis.
plt.xticks(rotation=20) # make labels readable.
plt.show() # display the plot.

▶ What you'll see: raw coefficients mostly reflect measurement units, not feature importance.

👀 Takeaway: standardize features before ridge unless the feature units are intentionally meaningful.

### Easy 4 — Validate λ on a held-out split

**Goal.** Choose λ with validation error, because training error alone prefers too little regularization.

In [ ]:
n_e4 = 60 # choose a modest sample size.
x1_e4 = np.linspace(-2, 2, n_e4) # first feature.
x2_e4 = x1_e4 + 0.1 * np.random.randn(n_e4) # correlated second feature.
X_e4 = np.column_stack([x1_e4, x2_e4]) # build the design matrix.
y_e4 = 2.5 * x1_e4 + 0.6 * np.random.randn(n_e4) # noisy target.
train_e4 = np.arange(n_e4) % 3 != 0 # deterministic two-thirds train split.
print("train size:", int(train_e4.sum()), "validation size:", int((~train_e4).sum())) # inspect split sizes.

▶ What you'll see: the data is split into training and validation examples.

In [ ]:
lams_e4 = np.array([0.0, 0.01, 0.1, 1.0, 10.0, 50.0]) # candidate lambdas.
val_mse_e4 = [] # store validation errors.
train_mse_e4 = [] # store training errors.
for lam_e4 in lams_e4: # evaluate each lambda.
    beta_e4 = ridge_beta(X_e4[train_e4], y_e4[train_e4], lam_e4) # fit on training data only.
    _, _, Xm_e4, ym_e4 = center_xy(X_e4[train_e4], y_e4[train_e4]) # recover training means for intercept.
    train_mse_e4.append(mse(y_e4[train_e4], ridge_predict(X_e4[train_e4], beta_e4, Xm_e4, ym_e4))) # training MSE.
    val_mse_e4.append(mse(y_e4[~train_e4], ridge_predict(X_e4[~train_e4], beta_e4, Xm_e4, ym_e4))) # validation MSE.
print("validation MSE:", np.round(val_mse_e4, 3)) # inspect validation scores.

In [ ]:
best_lam_e4 = float(lams_e4[int(np.argmin(val_mse_e4))]) # choose the lambda with lowest validation MSE.
print("best λ:", best_lam_e4) # inspect selected lambda.
plt.figure(figsize=(5, 3)) # create validation curve.
plt.semilogx(lams_e4 + 1e-6, train_mse_e4, marker="o", label="train") # plot training error.
plt.semilogx(lams_e4 + 1e-6, val_mse_e4, marker="o", label="validation") # plot validation error.
plt.axvline(best_lam_e4 + 1e-6, color="red", linestyle="--", label="best λ") # mark selected lambda.
plt.title("Easy 4: choose λ by validation") # title the plot.
plt.xlabel("λ") # label lambda axis.
plt.ylabel("MSE") # label error axis.
plt.legend() # show curve labels.
plt.show() # display the curve.

▶ What you'll see: validation, not training, identifies the preferred λ.

👀 Takeaway: ridge's penalty strength is a hyperparameter selected with unseen data.

### Easy 5 — Compare ridge to a flexible alternative

**Goal.** Recreate the lesson's gap arithmetic, because a flexible alternative must beat the stabilized full score by enough to matter.

In [ ]:
baseline_e5 = 0.368 # full score after adding ridge cost.
flexible_e5 = 0.416 # score for a tempting flexible alternative.
stable_e5 = 0.80 * baseline_e5 # stabilized score after a 20 percent reduction.
gap_e5 = flexible_e5 - baseline_e5 # compute absolute gap.
rel_gap_e5 = gap_e5 / flexible_e5 # compute relative gap.
print("gap:", round(gap_e5, 3), "relative:", round(rel_gap_e5, 3), "stable:", round(stable_e5, 3)) # inspect numbers.
assert round(gap_e5, 3) == 0.048 and round(stable_e5, 3) == 0.294 # verify lesson arithmetic.

▶ What you'll see: the absolute gap is 0.048 and the stabilized score is 0.294.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create comparison chart.
plt.bar(["baseline", "flexible", "stable"], [baseline_e5, flexible_e5, stable_e5], color=["gray", "crimson", "seagreen"]) # plot candidate scores.
plt.title("Easy 5: stabilized score wins") # title plot.
plt.ylabel("decision score") # label score axis.
plt.show() # display chart.

▶ What you'll see: the stabilized candidate is clearly lowest.

👀 Takeaway: selection should include penalty, validation gap, and practical decision scale.

## 🔴 Advanced

### Advanced 1 — Ridge reduces variance across resamples

**Goal.** Fit repeated noisy samples, because ridge's main benefit is often lower coefficient variance under collinearity.

In [ ]:
def center_xy(X, y): # center predictors and target inside the Advanced section.
    X = np.asarray(X, dtype=float) # convert predictors to floats.
    y = np.asarray(y, dtype=float) # convert targets to floats.
    return X - X.mean(axis=0), y - y.mean(), X.mean(axis=0), float(y.mean()) # return centered data and means.

def ridge_beta(X, y, lam): # solve centered ridge for advanced experiments.
    Xc, yc, _, _ = center_xy(X, y) # center before applying the penalty.
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc) # closed-form solution.

def ridge_predict(X, beta, X_mean, y_mean): # predict from centered coefficients.
    return (np.asarray(X, dtype=float) - X_mean) @ beta + y_mean # add the unpenalized intercept back.

def mse(y, pred): # compute validation MSE.
    return float(np.mean((np.asarray(y, dtype=float) - np.asarray(pred, dtype=float)) ** 2)) # average squared error.

runs_a1 = 80 # number of resampled datasets.
ols_coefs_a1 = [] # store OLS coefficients.
ridge_coefs_a1 = [] # store ridge coefficients.
base_x_a1 = np.linspace(-2, 2, 35) # fixed input grid.
for run_a1 in range(runs_a1): # repeat the experiment.
    rng_a1 = np.random.default_rng(run_a1) # deterministic local randomness.
    X_a1 = np.column_stack([base_x_a1, base_x_a1 + 0.08 * rng_a1.normal(size=base_x_a1.size)]) # correlated features.
    y_a1 = 3 * base_x_a1 + 0.8 * rng_a1.normal(size=base_x_a1.size) # noisy target.
    ols_coefs_a1.append(ridge_beta(X_a1, y_a1, 0.0)) # fit OLS.
    ridge_coefs_a1.append(ridge_beta(X_a1, y_a1, 2.0)) # fit ridge.
ols_coefs_a1 = np.array(ols_coefs_a1) # convert to matrix.
ridge_coefs_a1 = np.array(ridge_coefs_a1) # convert to matrix.
print("OLS coef std:", np.round(ols_coefs_a1.std(axis=0), 3)) # inspect OLS variance.
print("ridge coef std:", np.round(ridge_coefs_a1.std(axis=0), 3)) # inspect ridge variance.
assert ridge_coefs_a1.std() < ols_coefs_a1.std() # verify ridge is more stable overall.

▶ What you'll see: ridge coefficients vary less across resampled noisy datasets.

In [ ]:
plt.figure(figsize=(5, 3)) # create coefficient-cloud plot.
plt.scatter(ols_coefs_a1[:, 0], ols_coefs_a1[:, 1], s=18, alpha=0.6, label="OLS", color="crimson") # plot OLS coefficients.
plt.scatter(ridge_coefs_a1[:, 0], ridge_coefs_a1[:, 1], s=18, alpha=0.6, label="ridge", color="seagreen") # plot ridge coefficients.
plt.title("Advanced 1: ridge coefficient cloud is tighter") # title plot.
plt.xlabel("β1") # label first coefficient.
plt.ylabel("β2") # label second coefficient.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: the ridge cloud is pulled inward and is less spread out.

👀 Takeaway: ridge buys stability by reducing coefficient variance.

### Advanced 2 — Visualize the objective surface

**Goal.** Plot OLS and ridge objective contours, because the L2 penalty rounds out flat directions in coefficient space.

In [ ]:
b1_grid_a2 = np.linspace(-2, 5, 80) # grid for beta 1.
b2_grid_a2 = np.linspace(-2, 5, 80) # grid for beta 2.
B1_a2, B2_a2 = np.meshgrid(b1_grid_a2, b2_grid_a2) # create coefficient mesh.
X_a2 = np.column_stack([np.linspace(-1, 1, 25), np.linspace(-1, 1, 25) + 0.03 * np.random.randn(25)]) # correlated design.
y_a2 = 3 * X_a2[:, 0] + 0.1 * np.random.randn(25) # target.
Xc_a2, yc_a2, _, _ = center_xy(X_a2, y_a2) # center for slope objective.
print("grid shape:", B1_a2.shape) # inspect contour grid.

▶ What you'll see: a coefficient grid is ready for objective evaluation.

In [ ]:
obj_ols_a2 = np.zeros_like(B1_a2) # allocate OLS objective grid.
obj_ridge_a2 = np.zeros_like(B1_a2) # allocate ridge objective grid.
for i_a2 in range(B1_a2.shape[0]): # loop over rows of coefficient grid.
    for j_a2 in range(B1_a2.shape[1]): # loop over columns of coefficient grid.
        beta_a2 = np.array([B1_a2[i_a2, j_a2], B2_a2[i_a2, j_a2]]) # current coefficient pair.
        sse_a2 = np.sum((yc_a2 - Xc_a2 @ beta_a2) ** 2) # compute squared error.
        obj_ols_a2[i_a2, j_a2] = sse_a2 # store OLS objective.
        obj_ridge_a2[i_a2, j_a2] = sse_a2 + 2.0 * np.sum(beta_a2 ** 2) # store ridge objective.
print("min OLS objective:", round(float(obj_ols_a2.min()), 3)) # inspect minimum scale.

In [ ]:
fig_a2, ax_a2 = plt.subplots(1, 2, figsize=(8, 3)) # create side-by-side contours.
ax_a2[0].contour(B1_a2, B2_a2, obj_ols_a2, levels=18, cmap="Reds") # plot OLS contours.
ax_a2[0].set_title("OLS objective") # title first plot.
ax_a2[1].contour(B1_a2, B2_a2, obj_ridge_a2, levels=18, cmap="Greens") # plot ridge contours.
ax_a2[1].set_title("ridge objective") # title second plot.
for ax in ax_a2: # label both axes.
    ax.set_xlabel("β1") # x label.
    ax.set_ylabel("β2") # y label.
plt.tight_layout() # reduce overlap.
plt.show() # display contours.

▶ What you'll see: ridge contours are more centered and less willing to run far along flat directions.

👀 Takeaway: the L2 penalty reshapes the optimization landscape toward smaller, more stable coefficients.

### Advanced 3 — Use SVD to understand shrinkage by direction

**Goal.** Decompose ridge into singular directions, because weak data directions receive the strongest shrinkage.

In [ ]:
X_a3 = np.column_stack([np.linspace(-2, 2, 50), np.linspace(-2, 2, 50) + 0.05 * np.random.randn(50)]) # correlated design.
y_a3 = 3 * X_a3[:, 0] + 0.3 * np.random.randn(50) # target.
Xc_a3, yc_a3, _, _ = center_xy(X_a3, y_a3) # center data.
U_a3, s_a3, Vt_a3 = np.linalg.svd(Xc_a3, full_matrices=False) # decompose design into orthogonal directions.
print("singular values:", np.round(s_a3, 3)) # inspect strong and weak directions.
assert s_a3[0] > 10 * s_a3[1] # verify one direction is much weaker.

▶ What you'll see: one singular value is much smaller, indicating an unstable direction.

In [ ]:
lam_a3 = 1.0 # choose ridge strength.
shrink_a3 = s_a3 ** 2 / (s_a3 ** 2 + lam_a3) # ridge shrinkage factor per singular direction.
print("shrinkage factors:", np.round(shrink_a3, 3)) # inspect directional shrinkage.
assert shrink_a3[1] < shrink_a3[0] # verify the weak direction is shrunk more.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create shrinkage-factor plot.
plt.bar(["strong direction", "weak direction"], shrink_a3, color=["seagreen", "crimson"]) # compare directional shrinkage.
plt.ylim(0, 1.05) # keep factor scale bounded.
plt.title("Advanced 3: ridge shrinks weak directions most") # title plot.
plt.ylabel("s² / (s² + λ)") # label formula.
plt.xticks(rotation=12) # rotate labels.
plt.show() # display chart.

▶ What you'll see: the weak direction has a much smaller shrinkage factor.

👀 Takeaway: ridge is not uniform in prediction space; it targets directions the data supports poorly.

### Advanced 4 — Leave-one-out shortcut with the ridge hat matrix

**Goal.** Compute leave-one-out predictions from the hat matrix, because linear ridge models allow efficient validation diagnostics.

In [ ]:
x_a4 = np.linspace(-2, 2, 18) # small one-feature dataset.
X_a4 = np.column_stack([np.ones_like(x_a4), x_a4]) # include an explicit intercept column.
y_a4 = 1 + 2 * x_a4 + 0.4 * np.random.randn(18) # noisy linear targets.
lam_a4 = 1.0 # choose ridge penalty for the slope.
P_a4 = np.diag([0.0, 1.0]) # do not penalize intercept, penalize slope.
A_a4 = X_a4.T @ X_a4 + lam_a4 * P_a4 # build penalized normal-equation matrix.
beta_a4 = np.linalg.solve(A_a4, X_a4.T @ y_a4) # fit ridge with explicit intercept.
yhat_a4 = X_a4 @ beta_a4 # compute fitted values.
print("beta:", np.round(beta_a4, 3)) # inspect fitted intercept and slope.

▶ What you'll see: the intercept and slope are fitted while only the slope is penalized.

In [ ]:
H_a4 = X_a4 @ np.linalg.solve(A_a4, X_a4.T) # compute ridge hat matrix mapping y to fitted y.
h_a4 = np.diag(H_a4) # extract leverage values.
loo_pred_a4 = (yhat_a4 - h_a4 * y_a4) / (1 - h_a4) # linear-model leave-one-out prediction formula.
loo_mse_a4 = mse(y_a4, loo_pred_a4) # compute LOO MSE.
print("leverage range:", round(float(h_a4.min()), 3), "to", round(float(h_a4.max()), 3)) # inspect influence values.
print("LOO MSE:", round(loo_mse_a4, 3)) # inspect leave-one-out error.
assert np.all(h_a4 < 1) # verify formula denominator is safe.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create leverage plot.
plt.scatter(x_a4, h_a4, color="purple") # plot leverage by x position.
plt.title("Advanced 4: ridge leverage values") # title plot.
plt.xlabel("x") # label input axis.
plt.ylabel("hat diagonal hᵢ") # label leverage axis.
plt.show() # display plot.

▶ What you'll see: edge points usually have higher leverage, so leaving them out changes their predictions more.

👀 Takeaway: the ridge hat matrix exposes which observations strongly influence fitted values.

### Advanced 5 — Compare polynomial ridge models

**Goal.** Tune both polynomial degree and λ, because ridge is often paired with expanded features that need regularization.

In [ ]:
x_a5 = np.linspace(-1, 1, 80) # input grid.
y_true_a5 = np.sin(3 * x_a5) # smooth nonlinear signal.
y_a5 = y_true_a5 + 0.15 * np.random.randn(80) # noisy observations.
train_a5 = np.arange(80) % 4 != 0 # deterministic train/validation split.
degrees_a5 = np.array([1, 3, 7]) # candidate polynomial degrees.
lams_a5 = np.array([0.0, 0.01, 0.1, 1.0, 10.0]) # candidate penalties.
print("degrees:", degrees_a5, "lambdas:", lams_a5) # inspect grid.

▶ What you'll see: a small hyperparameter grid combines flexibility and regularization.

In [ ]:
def poly_features_a5(x, degree): # build polynomial features without an intercept column.
    return np.column_stack([x ** p for p in range(1, degree + 1)]) # powers x, x^2, ..., x^degree.

results_a5 = [] # store (degree, lambda, validation MSE).
for degree_a5 in degrees_a5: # loop over polynomial capacities.
    X_all_a5 = poly_features_a5(x_a5, int(degree_a5)) # construct features for this degree.
    scale_a5 = X_all_a5[train_a5].std(axis=0) # compute training feature scales.
    scale_a5[scale_a5 == 0] = 1.0 # guard against zero scale.
    X_all_a5 = (X_all_a5 - X_all_a5[train_a5].mean(axis=0)) / scale_a5 # standardize with training statistics.
    for lam_grid_a5 in lams_a5: # loop over ridge strengths.
        beta_a5 = ridge_beta(X_all_a5[train_a5], y_a5[train_a5], float(lam_grid_a5)) # fit training ridge.
        _, _, Xm_fit_a5, ym_fit_a5 = center_xy(X_all_a5[train_a5], y_a5[train_a5]) # get intercept means.
        pred_val_a5 = ridge_predict(X_all_a5[~train_a5], beta_a5, Xm_fit_a5, ym_fit_a5) # predict validation points.
        results_a5.append((degree_a5, lam_grid_a5, mse(y_a5[~train_a5], pred_val_a5))) # store validation error.
print("number of models:", len(results_a5)) # inspect grid size.

In [ ]:
best_a5 = min(results_a5, key=lambda row: row[2]) # choose lowest validation MSE.
print("best degree, λ, val MSE:", best_a5[0], best_a5[1], round(best_a5[2], 3)) # inspect best configuration.
assert best_a5[2] < 0.2 # verify a reasonable nonlinear fit was found.

In [ ]:
plt.figure(figsize=(5, 3)) # create validation curves by degree.
for degree_a5 in degrees_a5: # plot one curve per degree.
    vals_a5 = [row[2] for row in results_a5 if row[0] == degree_a5] # collect errors for this degree.
    plt.semilogx(lams_a5 + 1e-6, vals_a5, marker="o", label=f"degree {degree_a5}") # plot validation MSE.
plt.title("Advanced 5: degree and λ validation grid") # title plot.
plt.xlabel("λ") # label penalty axis.
plt.ylabel("validation MSE") # label error axis.
plt.legend() # show degree labels.
plt.show() # display curves.

▶ What you'll see: higher-degree models need ridge to avoid chasing noise, while too much λ underfits.

👀 Takeaway: ridge lets flexible feature maps generalize by charging the coefficient length they require.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Ridge buys stability by charging squared coefficient length.

Ridge regression keeps least squares but adds an L2 penalty. That penalty is the practical bridge from empirical risk to stability: it discourages large coefficients when features are noisy, correlated, or too numerous for the sample size.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)
random.seed(7)

from sklearn.linear_model import Ridge

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs

def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))

def linear_baseline(x_tr, y_tr, x_te):
    clf = LinearRegression()
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)

## The concept, built once on D1

The lesson formula is

$$\hat\beta=\arg\min_\beta \|y-X\beta\|_2^2+\lambda\|\beta\|_2^2$$

First we reproduce the lesson arithmetic exactly: the three cited losses are 0.268, 0.122, and 0.505.

In [ ]:
lesson_losses = np.array([0.268, 0.122, 0.505])
lesson_total = round(float(lesson_losses.sum()), 3)
lesson_risk = round(lesson_total / 3.0, 3)
lesson_cost = 0.07
lesson_score = round(lesson_risk + lesson_cost, 3)
lesson_alternative = 0.416
lesson_gap = round(lesson_alternative - lesson_score, 3)
lesson_stabilized = round(0.8 * lesson_score, 3)

assert lesson_total == 0.895
assert lesson_risk == 0.298
assert lesson_score == 0.368
assert lesson_gap == 0.048
assert lesson_stabilized == 0.294

print("loss total", lesson_total)
print("empirical risk", lesson_risk)
print("cost-adjusted score", lesson_score)
print("validation gap", lesson_gap)
print("stabilized score", lesson_stabilized)

Now we package the real estimator as `ridge_regression_l2_method()`. The method is reusable: it accepts training data and returns predictions for new rows, so the exact same call can run from D1 through D5.

In [ ]:
def ridge_regression_l2_method(alpha=10.0):
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha),
    )

    def build_and_predict(x_tr, y_tr, x_te):
        model.fit(x_tr, y_tr)
        return model.predict(x_te)

    return build_and_predict

## The dataset ladder

All six notebooks in this batch use `reg_ladder()`: D1 is inspectable, D2 is clean linear signal, D3 is nonlinear sine signal, D4 is sklearn's real diabetes regression dataset, and D5 is a real high-dimensional diabetes interaction design built without downloads. We report MSE as the lesson-plan metric, with `reg_rmse()` as the shared helper check and R² as a secondary annotation.

In [ ]:
rungs = reg_ladder()
diabetes = load_diabetes()
interaction_builder = PolynomialFeatures(degree=2, include_bias=False)
real_d5_X = interaction_builder.fit_transform(diabetes.data)
real_d5_y = diabetes.target
rungs[-1] = ("D5 Diabetes interactions (real, 65-D)", real_d5_X, real_d5_y)

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    print(f"{rung_index}. {name}")
    print("  X shape", X.shape)
    print("  y size", y.shape[0])
    print("  X sample", np.round(X[:3], 3).tolist())
    print("  y sample", np.round(y[:3], 3).tolist())

## Run the same method across D1–D5

We compare L2 shrinkage vs unregularized least squares. The no-skill baseline is `linear_baseline`; the lesson method is `ridge_regression_l2_method`. `reg_rmse()` is called on every rung to keep this regression batch tied to the shared helper.

In [ ]:
method = ridge_regression_l2_method(alpha=1.0)
baseline = linear_baseline
rows = []
prediction_store = []

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    baseline_pred = baseline(x_tr, y_tr, x_te)
    method_pred = method(x_tr, y_tr, x_te)
    baseline_mse = float(mean_squared_error(y_te, baseline_pred))
    method_mse = float(mean_squared_error(y_te, method_pred))
    baseline_rmse = reg_rmse(baseline, X, y)
    method_rmse = reg_rmse(method, X, y)
    assert abs(method_rmse - math.sqrt(method_mse)) < 1e-8
    method_r2 = float(r2_score(y_te, method_pred))
    rows.append((rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2))
    prediction_store.append((name, X, y, x_te, y_te, method_pred))

print("rung | dataset | linear RMSE | method RMSE | method MSE | method R^2")
for rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2 in rows:
    print(f"D{rung_index} | {name} | {baseline_rmse:.3f} | {method_rmse:.3f} | {method_mse:.3f} | {method_r2:.3f}")

## Results visualization

The closing figure has two parts: small multiples show the fitted output artifact on each rung, and the summary curve tracks MSE from D1 to D5.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for col, (name, X, y, x_te, y_te, method_pred) in enumerate(prediction_store):
    axis = axes[0, col]
    order = np.argsort(x_te[:, 0])
    axis.scatter(x_te[:, 0], y_te, s=18, alpha=0.65, label="truth")
    axis.scatter(x_te[:, 0], method_pred, s=18, alpha=0.65, label="prediction")
    axis.set_title(f"D{col + 1} ridge-shrunk linear fit", fontsize=9)
    axis.set_xlabel("first feature")
    if col == 0:
        axis.set_ylabel("target")
        axis.legend(fontsize=8)

mse_values = [row[4] for row in rows]
axes[1, 0].plot(range(1, 6), mse_values, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("MSE")
axes[1, 0].set_title("MSE vs ladder complexity")

for empty_axis in axes[1, 1:]:
    empty_axis.axis("off")

plt.tight_layout()
plt.show()

## Pitfall on D5: optimizing the raw term and forgetting the cost

The lesson warns that raw empirical risk is not the whole selection score. On D5 we reproduce that mistake, then add the lesson cost/scale/gap check before choosing the winner.

In [ ]:
d5_name, d5_X, d5_y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(d5_X, d5_y, test_size=0.4, random_state=0)
method_pred = method(x_tr, y_tr, x_te)
baseline_pred = baseline(x_tr, y_tr, x_te)
method_raw = math.sqrt(float(mean_squared_error(y_te, method_pred)))
baseline_raw = math.sqrt(float(mean_squared_error(y_te, baseline_pred)))
wrong_winner = "method" if method_raw < baseline_raw else "linear baseline"
scale = max(method_raw, baseline_raw, 1.0)
method_score = method_raw / scale + lesson_cost
baseline_score = baseline_raw / scale
fixed_winner = "method" if method_score < baseline_score else "linear baseline"
observed_gap = abs(method_score - baseline_score)

print("D5", d5_name)
print("wrong raw-only winner", wrong_winner)
print("method raw RMSE", round(method_raw, 3))
print("baseline raw RMSE", round(baseline_raw, 3))
print("lesson cost", lesson_cost)
print("cost-adjusted method score", round(method_score, 3))
print("cost-adjusted baseline score", round(baseline_score, 3))
print("fixed winner", fixed_winner)
print("observed adjusted gap", round(observed_gap, 3))
print("lesson minimum meaningful gap", lesson_gap)

if observed_gap < lesson_gap:
    print("decision: gap is too small; prefer the simpler setting or collect more validation data")
else:
    print("decision: adjusted gap clears the lesson check")

## Evaluate it + Practice

- Metric: MSE is the lesson-plan metric; `reg_rmse()` supplies the shared RMSE helper check and R² is secondary.
- No-skill baseline: compare against unregularized `linear_baseline` on every rung.
- Cheap sanity check: D1 should be explainable from the printed predictions; if it behaves oddly, inspect preprocessing, extrapolation, and the intercept.
- Ablation: turn off the key idea (L2 shrinkage vs unregularized least squares) and confirm the score or stability worsens on at least one harder rung.
- Failure signals: a tiny validation gap, scale-mismatched scores, or a D5 winner that changes after adding the lesson cost.

Practice 1: change one hyperparameter and rerun the ladder table. Which rung moves the most?

Practice 2: replace RMSE with MAE for the table. Does the D5 winner change?

Practice 3: add a short note explaining whether the lesson gap is large enough for deployment.